### Section 1: Database Creation & Preparation

In [1]:
import pandas as pd
from pathlib import Path

In [ ]:
file_name = "Aliaa_Sakr_BI_task_data.xlsx"

df = pd.read_excel(file_name)

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Rows: 1470
Columns: 35


In [ ]:
# Remove trailing spaces from text columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

In [4]:
# Convert binary columns

df["Attrition"] = df["Attrition"].map({
    "Yes":1,
    "No":0
})

df["OverTime"] = df["OverTime"].map({
    "Yes":1,
    "No":0
})

In [ ]:
# FUNCTION TO CREATE DIMENSION TABLES

def create_dimension(df, column_name, id_name):

    dimension = (
        df[[column_name]]
        .drop_duplicates()
        .sort_values(column_name)
        .reset_index(drop=True)
    )

    dimension[id_name] = range(1, len(dimension)+1)

    dimension = dimension[[id_name, column_name]]

    df = df.merge(
        dimension,
        on=column_name,
        how="left"
    )

    return df, dimension

In [ ]:
# CREATE DIMENSIONS

df, dim_education_field = create_dimension(
    df,
    "EducationField",
    "EducationFieldID"
)

df, dim_business_travel = create_dimension(
    df,
    "BusinessTravel",
    "BusinessTravelID"
)

df, dim_department = create_dimension(
    df,
    "Department",
    "DepartmentID"
)

df, dim_gender = create_dimension(
    df,
    "Gender",
    "GenderID"
)

df, dim_job_role = create_dimension(
    df,
    "JobRole",
    "JobRoleID"
)

df, dim_marital_status = create_dimension(
    df,
    "MaritalStatus",
    "MaritalStatusID"
)


In [ ]:
# CREATE FACT TABLE

fact = df.drop(columns=[
    # Dimension columns
    "EducationField",
    "BusinessTravel",
    "Department",
    "Gender",
    "JobRole",
    "MaritalStatus",

    # Removed unnecessary columns
    "DailyRate",
    "HourlyRate",
    "MonthlyRate",
    "EmployeeCount",
    "Over18",
    "StandardHours"
])

# Arrange columns

fact_columns = [

    "EmployeeNumber",

    "EducationFieldID",
    "BusinessTravelID",
    "DepartmentID",
    "GenderID",
    "JobRoleID",
    "MaritalStatusID",

    "Age",
    "Attrition",
    "DistanceFromHome",
    "Education",
    "EnvironmentSatisfaction",
    "JobInvolvement",
    "JobLevel",
    "JobSatisfaction",
    "MonthlyIncome",
    "NumCompaniesWorked",
    "OverTime",
    "PercentSalaryHike",
    "PerformanceRating",
    "RelationshipSatisfaction",
    "StockOptionLevel",
    "TotalWorkingYears",
    "TrainingTimesLastYear",
    "WorkLifeBalance",
    "YearsAtCompany",
    "YearsInCurrentRole",
    "YearsSinceLastPromotion",
    "YearsWithCurrManager"

]

fact = fact[fact_columns]

In [8]:
fact.insert(0, "EmployeeKey", range(1, len(fact) + 1))

In [ ]:
# EXPORT STAR SCHEMA

output_folder = Path("HR_Star_Schema")

output_folder.mkdir(exist_ok=True)


dim_education_field.to_excel(
    output_folder / "Dim_EducationField.xlsx",
    index=False
)

dim_business_travel.to_excel(
    output_folder / "Dim_BusinessTravel.xlsx",
    index=False
)

dim_department.to_excel(
    output_folder / "Dim_Department.xlsx",
    index=False
)

dim_gender.to_excel(
    output_folder / "Dim_Gender.xlsx",
    index=False
)

dim_job_role.to_excel(
    output_folder / "Dim_JobRole.xlsx",
    index=False
)

dim_marital_status.to_excel(
    output_folder / "Dim_MaritalStatus.xlsx",
    index=False
)

fact.to_excel(
    output_folder / "Fact_Employees.xlsx",
    index=False
)

print("\nStar Schema Successfully Created!")

print(f"\nFiles exported to:\n{output_folder.resolve()}")

print("\nCreated Tables:")

print("- Fact_Employees")
print("- Dim_EducationField")
print("- Dim_BusinessTravel")
print("- Dim_Department")
print("- Dim_Gender")
print("- Dim_JobRole")
print("- Dim_MaritalStatus")


Star Schema Successfully Created!

Files exported to:
C:\Aliaa\work\my_work\2026\Projects\TP\HR_Star_Schema

Created Tables:
- Fact_Employees
- Dim_EducationField
- Dim_BusinessTravel
- Dim_Department
- Dim_Gender
- Dim_JobRole
- Dim_MaritalStatus


### Section 2: Request number 8

In [ ]:
# Request # 8
# CLASSIFY EMPLOYEES BY EXPERIENCE LEVEL

def classify_experience(years):
    if years < 5:
        return "Junior"
    elif years < 10:
        return "Mid"
    else:
        return "Senior"

fact["ExperienceLevel"] = fact["TotalWorkingYears"].apply(classify_experience)

# Summary table
summary = (
    fact.groupby("ExperienceLevel")
    .size()
    .reset_index(name="EmployeeCount")
)

# Enforce logical order instead of alphabetical
order = ["Junior", "Mid", "Senior"]
summary["ExperienceLevel"] = pd.Categorical(summary["ExperienceLevel"], categories=order, ordered=True)
summary = summary.sort_values("ExperienceLevel").reset_index(drop=True)

print(summary)

  ExperienceLevel  EmployeeCount
0          Junior            228
1             Mid            493
2          Senior            749


### Section 3: SQL database creation

In [ ]:
from sqlalchemy import create_engine
import urllib

In [ ]:
server = "DESKTOP-2APG0PN\\SQLEXPRESS01" 
database = "HR_Star_Schema"

params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

fact.to_sql("Fact_Employees", engine, if_exists="replace", index=False)
dim_education_field.to_sql("Dim_EducationField", engine, if_exists="replace", index=False)
dim_business_travel.to_sql("Dim_BusinessTravel", engine, if_exists="replace", index=False)
dim_department.to_sql("Dim_Department", engine, if_exists="replace", index=False)
dim_gender.to_sql("Dim_Gender", engine, if_exists="replace", index=False)
dim_job_role.to_sql("Dim_JobRole", engine, if_exists="replace", index=False)
dim_marital_status.to_sql("Dim_MaritalStatus", engine, if_exists="replace", index=False)

print("Tables loaded into SQL Server successfully.")

Tables loaded into SQL Server successfully.
